MCP Client

In [ ]:
def as_markdown(value) -> Markdown:
    # MCP ToolResult / ResourceResult möglichst lesbar ausgeben
    if hasattr(value, "content"):
        parts = []

        for item in value.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            else:
                parts.append(str(item))

        text = "\n\n".join(parts)
        return Markdown(f"```text\n{text}\n```")

    if hasattr(value, "contents"):
        parts = []

        for item in value.contents:
            if hasattr(item, "text"):
                parts.append(item.text)
            elif hasattr(item, "blob"):
                parts.append(str(item.blob))
            else:
                parts.append(str(item))

        text = "\n\n".join(parts)
        return Markdown(f"```text\n{text}\n```")

    if isinstance(value, (dict, list)):
        return Markdown(
            "```json\n"
            + json.dumps(value, indent=2, ensure_ascii=False)
            + "\n```"
        )

    return Markdown(f"```text\n{value}\n```")

In [ ]:
from mcp.client.sse import sse_client
from mcp import ClientSession

import json
import shlex
import asyncio
from IPython.display import display, Markdown
from IPython.core.magic import Magics, magics_class, cell_magic

MCP_URL = "http://localhost:8000/sse"


async def _mcp_execute(line: str, cell: str = ""):
    args = shlex.split(line)

    if not args:
        raise ValueError(
            "Usage: %%mcp tools | %%mcp resources | %%mcp resource <uri> | %%mcp tool <name>"
        )

    command = args[0]

    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            if command == "tools":
                return await session.list_tools()

            if command == "resources":
                return await session.list_resources()

            if command == "resource":
                if len(args) < 2:
                    raise ValueError("Usage: %%mcp resource <uri>")
                return await session.read_resource(args[1])

            if command == "tool":
                if len(args) < 2:
                    raise ValueError("Usage: %%mcp tool <tool_name>")

                tool_name = args[1]
                payload = json.loads(cell) if cell.strip() else {}

                return await session.call_tool(tool_name, payload)

            raise ValueError(f"Unknown MCP command: {command}")


@magics_class
class MCPMagics(Magics):

    @cell_magic
    def mcp(self, line, cell):
        async def runner():
            try:
                result = await _mcp_execute(line, cell)
                display(as_markdown(result))
            except Exception as exc:
                display(Markdown(
                    f"```text\n{type(exc).__name__}: {exc}\n```"
                ))

        loop = asyncio.get_running_loop()
        task = loop.create_task(runner())

        return task

ip = get_ipython()

# Vorherige Version sauber entfernen, falls mehrfach ausgeführt
if "mcp" in ip.magics_manager.magics["cell"]:
    del ip.magics_manager.magics["cell"]["mcp"]

ip.register_magics(MCPMagics)

In [ ]:
%%mcp tools
{}

In [ ]:
%%mcp tool pwd
{}

In [ ]:
%%mcp tool df
{}